In [1]:
!pip install cudf-cu11 cuml-cu11 --extra-index-url=https://pypi.nvidia.com
# =============================================================================
# IMPORTS
# =============================================================================
import os
import platform
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report, roc_auc_score, roc_curve, precision_recall_curve, auc, average_precision_score
from sklearn.cluster import KMeans
from scipy.stats import beta as beta_dist
import joblib
import time
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

# =============================================================================
# HARDWARE DETECTION & ACCELERATOR SETUP
# =============================================================================
print("\n" + "="*80)
print("HARDWARE DETECTION & ACCELERATOR SETUP")
print("="*80)
print(f"Platform: {platform.platform()}")

# GPU Detection and cuML setup for accelerated sklearn
USE_GPU = False
GPU_AVAILABLE = False

try:
    # RAPIDS (cuDF/cuML) requires NVIDIA Volta (Compute Capability >= 7.0).
    # Some common notebook GPUs (e.g., Tesla P100, CC 6.0) can run CUDA/CuPy
    # but cannot run cuDF/cuML.
    import cupy as cp

    try:
        props = cp.cuda.runtime.getDeviceProperties(0)
        cc_major = int(props.get('major', -1))
        cc_minor = int(props.get('minor', -1))
    except Exception:
        cc_major, cc_minor = -1, -1

    if cc_major != -1 and cc_major < 7:
        raise RuntimeError(
            f"RAPIDS requires Compute Capability >= 7.0, detected {cc_major}.{cc_minor}. "
            "Falling back to CPU sklearn."
        )

    import cudf
    from cuml.ensemble import RandomForestClassifier as cuRF
    from cuml.linear_model import LogisticRegression as cuLR
    from cuml.naive_bayes import GaussianNB as cuNB
    from cuml.tree import DecisionTreeClassifier as cuDT
    from cuml.cluster import KMeans as cuKMeans
    from cuml.neural_network import MLPClassifier as cuMLP
    GPU_AVAILABLE = True
    USE_GPU = True
    
    # Get GPU info
    import subprocess
    result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], 
                          capture_output=True, text=True)
    gpu_info = result.stdout.strip()
    print(f"\n✅ GPU Acceleration Available (RAPIDS cuML)")
    print(f"   GPU: {gpu_info}")
    print(f"   cuML will be used for accelerated training")
except Exception as e:
    # Covers ImportError, cuDF UnsupportedCUDAError, and compute capability checks.
    print("\n⚠️ RAPIDS cuML not available - using CPU sklearn")
    print(f"   Reason: {type(e).__name__}: {e}")
    print("   If you want RAPIDS GPU acceleration: use a Volta+ GPU (T4/V100/A100, CC>=7.0).")
    print("   Otherwise: keep sklearn/pandas (CPU) and everything will still run.")

# Fallback to sklearn
from sklearn.neural_network import MLPClassifier as skMLP
from sklearn.ensemble import RandomForestClassifier as skRF
from sklearn.tree import DecisionTreeClassifier as skDT
from sklearn.naive_bayes import GaussianNB as skNB
from sklearn.linear_model import LogisticRegression as skLR

# Set model classes based on availability
if USE_GPU:
    print("\n🚀 Using GPU-accelerated models (cuML)")
    # Note: cuML MLP might not be available, fall back to sklearn
    try:
        MLPClassifier = cuMLP
    except:
        MLPClassifier = skMLP
        print("   ⚠️ cuML MLP not available, using sklearn MLP")
    RandomForestClassifier = cuRF
    DecisionTreeClassifier = cuDT
    GaussianNB = cuNB
    LogisticRegression = cuLR
    KMeansCluster = cuKMeans
else:
    print("\n🔧 Using CPU models (sklearn)")
    MLPClassifier = skMLP
    RandomForestClassifier = skRF
    DecisionTreeClassifier = skDT
    GaussianNB = skNB
    LogisticRegression = skLR
    KMeansCluster = KMeans

print("="*80 + "\n")

# =============================================================================
# CONFIGURATION
# =============================================================================

# Dataset
DATA_DIR = "/kaggle/input/cic-ids2017/MachineLearningCVE"
CSV_FILES = [
    "Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv",
    "Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv",
    "Friday-WorkingHours-Morning.pcap_ISCX.csv",
    "Monday-WorkingHours.pcap_ISCX.csv",
    "Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv",
    "Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv",
    "Tuesday-WorkingHours.pcap_ISCX.csv",
]

# Output
OUT_DIR = "/kaggle/working/models"
PLOT_DIR = "/kaggle/working/plots"
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(PLOT_DIR, exist_ok=True)

# Set seed for reproducibility
seed = 42
np.random.seed(seed)


# =============================================================================
# MODEL SELECTION - Comment/uncomment to choose which models to train
# =============================================================================
MODELS_TO_TRAIN = {
    'mlp_model': True,          # ✅ MLP Classifier
    'nb_model': True,           # ✅ Naive Bayes
    'rf_model': True,           # ✅ Random Forest
    'dt_model': True,           # ✅ Decision Tree
    'lr_model': True,           # ✅ Logistic Regression
    'apollon_model': True,      # ✅ Apollon (MAB with Thompson Sampling)
}


# =============================================================================
# APOLLON ARCHITECTURE (Multi-Armed Bandit with Thompson Sampling)
# =============================================================================

class MultiArmedBanditThompsonSampling:
    """
    Apollon: A Robust Defence System against Adversarial Machine Learning Attacks in IDS
    
    This implementation uses Multi-Armed Bandits (MAB) with Thompson Sampling to dynamically
    select the optimal classifier or ensemble of classifiers for each input. This approach
    prevents attackers from learning the IDS behaviour and generating adversarial examples.
    
    Reference: Apollon paper - "Apollon: A Robust Defence System against Adversarial 
    Machine Learning Attacks in Intrusion Detection Systems"
    """
    
    def __init__(self, n_arms=5, n_clusters=2, seed=42):
        """
        Initialize the Multi-Armed Bandit with Thompson Sampling.
        
        Parameters:
            n_arms: Number of classifier arms (default: 5 - RF, DT, NB, LR, MLP)
            n_clusters: Number of clusters for input space partitioning
            seed: Random seed for reproducibility
        """
        self.n_arms = n_arms
        self.n_clusters = n_clusters
        self.seed = seed
        self.use_gpu = USE_GPU
        np.random.seed(seed)
        
        # Initialize classifier arms (use GPU-accelerated versions if available)
        if USE_GPU:
            self.arms = [
                RandomForestClassifier(n_estimators=100, random_state=seed),
                DecisionTreeClassifier(random_state=seed),
                GaussianNB(),
                LogisticRegression(max_iter=1000),
                skMLP(hidden_layer_sizes=(32,), max_iter=200, random_state=seed, 
                     batch_size=200, early_stopping=True, activation='tanh', solver='adam')
            ]
        else:
            self.arms = [
                RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=seed),
                DecisionTreeClassifier(random_state=seed),
                GaussianNB(),
                LogisticRegression(max_iter=1000, random_state=seed),
                MLPClassifier(hidden_layer_sizes=(32,), max_iter=200, random_state=seed, 
                             batch_size=200, early_stopping=True, activation='tanh', solver='adam')
            ]
        self.arm_names = ['RandomForest', 'DecisionTree', 'NaiveBayes', 'LogisticRegression', 'MLP']
        
        self.cluster_centers = None
        self.cluster_assignments = None
        self.reward_sums = {}
        for cluster in range(n_clusters):
            self.reward_sums[cluster] = np.zeros(n_arms)
        
        # Beta distribution parameters for Thompson Sampling
        self.alpha = np.ones(self.n_arms)
        self.beta = np.ones(self.n_arms)
        
        self.training_time = 0.0
    
    def _to_numpy(self, arr):
        """Convert cupy/cudf arrays to numpy if needed."""
        if USE_GPU:
            try:
                if hasattr(arr, 'get'):
                    return arr.get()
                elif hasattr(arr, 'to_numpy'):
                    return arr.to_numpy()
            except:
                pass
        return np.asarray(arr)
    
    def train(self, X_train, y_train):
        """
        Train all classifier arms and cluster the input space.
        
        Parameters:
            X_train: Training features
            y_train: Training labels
        """
        start_time = time.time()
        
        # Cluster the training data (use GPU KMeans if available)
        print(f"    Clustering input space into {self.n_clusters} clusters...")
        kmeans = KMeansCluster(n_clusters=self.n_clusters, random_state=self.seed, n_init=10)
        self.cluster_assignments = self._to_numpy(kmeans.fit_predict(X_train))
        self.cluster_centers = self._to_numpy(kmeans.cluster_centers_)
        
        # Print cluster distribution
        for i in range(self.n_clusters):
            cluster_count = np.sum(self.cluster_assignments == i)
            print(f"      Cluster {i}: {cluster_count:,} samples ({cluster_count/len(y_train)*100:.2f}%)")
        
        # Train all arms on the full training data
        print(f"\n    Training {self.n_arms} classifier arms...")
        for arm_idx, (arm, name) in enumerate(zip(self.arms, self.arm_names)):
            print(f"      [{arm_idx+1}/{self.n_arms}] Training {name}...")
            arm.fit(X_train, y_train)
        
        # Calculate reward sums for each arm in each cluster
        print(f"\n    Computing arm rewards per cluster...")
        for cluster_idx in range(self.n_clusters):
            cluster_mask = self.cluster_assignments == cluster_idx
            cluster_X = X_train[cluster_mask]
            cluster_y = y_train[cluster_mask]
            
            for arm_idx, arm in enumerate(self.arms):
                if len(cluster_X) > 0:
                    arm_preds = arm.predict(cluster_X)
                    accuracy = np.mean(arm_preds == cluster_y)
                    self.reward_sums[cluster_idx][arm_idx] = accuracy
                    print(f"      Cluster {cluster_idx}, {self.arm_names[arm_idx]}: {accuracy*100:.2f}% accuracy")
        
        self.training_time = time.time() - start_time
        print(f"\n    ✅ Training completed in {self.training_time:.2f} seconds")
    
    def select_arm(self, cluster):
        """
        Select an arm using Thompson Sampling for the given cluster.
        
        Parameters:
            cluster: Cluster index
            
        Returns:
            Selected arm index
        """
        theta = np.zeros(self.n_arms)
        for arm_idx in range(self.n_arms):
            # Sample from Beta distribution
            alpha_param = self.alpha[arm_idx] + self.reward_sums[cluster][arm_idx]
            beta_param = self.beta[arm_idx] + 1 - self.reward_sums[cluster][arm_idx]
            theta[arm_idx] = np.random.beta(alpha_param, beta_param)
        return np.argmax(theta)
    
    def predict(self, X_test):
        """
        Predict using Thompson Sampling to dynamically select classifiers.
        
        Parameters:
            X_test: Test features
            
        Returns:
            predictions: Predicted labels
            selected_arms: Arms selected for each sample
        """
        # Assign each sample to a cluster
        clusters = np.zeros(len(X_test), dtype=int)
        for i in range(len(X_test)):
            distances = np.linalg.norm(self.cluster_centers - X_test[i], axis=1)
            clusters[i] = np.argmin(distances)
        
        # Select arm for each sample using Thompson Sampling
        selected_arms = np.zeros(len(X_test), dtype=int)
        for i in range(len(X_test)):
            selected_arms[i] = self.select_arm(clusters[i])
        
        # Predict using selected arms
        predictions = np.zeros(len(X_test), dtype=int)
        for arm_idx in range(self.n_arms):
            arm_mask = selected_arms == arm_idx
            if np.sum(arm_mask) > 0:
                arm_X = X_test[arm_mask]
                predictions[arm_mask] = self.arms[arm_idx].predict(arm_X)
        
        return predictions, selected_arms
    
    def predict_proba(self, X_test):
        """
        Predict probabilities using Thompson Sampling.
        
        Parameters:
            X_test: Test features
            
        Returns:
            probabilities: Probability of positive class (Attack)
        """
        predictions, selected_arms = self.predict(X_test)
        
        # Get probabilities from each arm
        probabilities = np.zeros(len(X_test))
        for arm_idx in range(self.n_arms):
            arm_mask = selected_arms == arm_idx
            if np.sum(arm_mask) > 0:
                arm_X = X_test[arm_mask]
                if hasattr(self.arms[arm_idx], 'predict_proba'):
                    probs = self.arms[arm_idx].predict_proba(arm_X)
                    if probs.shape[1] > 1:
                        probabilities[arm_mask] = probs[:, 1]
                    else:
                        probabilities[arm_mask] = probs[:, 0]
                else:
                    # For classifiers without predict_proba, use predictions
                    probabilities[arm_mask] = predictions[arm_mask]
        
        return probabilities

# =============================================================================
# DATA LOADING AND PREPROCESSING
# =============================================================================
print("\n" + "="*80)
print("DATA LOADING AND PREPROCESSING")
print("="*80)

# Metadata columns to drop
drop_columns = [
    "Flow ID",    
    'Fwd Header Length.1',
    "Source IP", "Src IP",
    "Source Port", "Src Port",
    "Destination IP", "Dst IP",
    "Destination Port", "Dst Port",
    "Timestamp",
]

# Load CSV files
print(f"Loading {len(CSV_FILES)} CSV files from {DATA_DIR}...")
individual_dfs = []
for idx, csv_file in enumerate(CSV_FILES, 1):
    file_path = os.path.join(DATA_DIR, csv_file)
    print(f"  [{idx}/{len(CSV_FILES)}] Loading: {csv_file}")
    df = pd.read_csv(file_path, sep=',', encoding='utf-8')
    print(f"      Initial shape: {df.shape}")
    print(f"      Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
    individual_dfs.append(df)

print(f"\nTotal DataFrames loaded: {len(individual_dfs)}")
print(f"Total memory before optimization: {sum(df.memory_usage(deep=True).sum() for df in individual_dfs) / 1024**2:.2f} MB")

# Preprocess each dataframe
print("\nPreprocessing individual DataFrames...")
print("-" * 80)
for i, df in enumerate(individual_dfs):
    print(f"\nDataFrame {i} ({CSV_FILES[i]}):")
    
    # Strip whitespace from column names
    df.columns = df.columns.str.strip()
    print(f"  Total columns: {len(df.columns)}")
    
    # Drop metadata columns
    dropped = [col for col in drop_columns if col in df.columns]
    df.drop(columns=drop_columns, inplace=True, errors='ignore')
    if dropped:
        print(f"  Dropped metadata columns: {len(dropped)}")
    
    # Standardize label naming
    df['Label'] = df['Label'].replace({'BENIGN': 'Benign'})
    
    # Replace infinite values with NaN
    inf_count = np.isinf(df.select_dtypes(include=[np.number]).values).sum()
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    if inf_count > 0:
        print(f"  Infinite values replaced with NaN: {inf_count}")
    
    # Drop NaN values
    nan_rows = df.isna().any(axis=1).sum()
    df.dropna(inplace=True)
    if nan_rows > 0:
        print(f"  Rows with NaN removed: {nan_rows}")
    
    # Drop duplicates
    duplicates = df.duplicated().sum()
    df.drop_duplicates(inplace=True)
    if duplicates > 0:
        print(f"  Duplicate rows removed: {duplicates}")
    
    df.reset_index(drop=True, inplace=True)
    print(f"  Final shape: {df.shape}")
    print(f"  Memory: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

# Concatenate all dataframes
print("\nConcatenating DataFrames...")
df_data = pd.concat(individual_dfs, axis=0, ignore_index=True)
print(f"Combined dataset shape: {df_data.shape}")

# Additional cleaning
print("\nFinal data cleaning...")
null_counts = df_data.isnull().sum()
print(f"  Null entries found: {null_counts.sum()}")
if null_counts.sum() > 0:
    df_data.dropna(inplace=True)
    print(f"  After removing nulls: {df_data.shape}")

duplicate_count = df_data.duplicated().sum()
print(f"  Duplicate entries found: {duplicate_count}")
if duplicate_count > 0:
    df_data.drop_duplicates(inplace=True)
    print(f"  After removing duplicates: {df_data.shape}")

df_data.reset_index(drop=True, inplace=True)

# Inspect label distribution
print("\n" + "-"*80)
print("LABEL DISTRIBUTION (Before Binarization)")
print("-"*80)
print("\nAbsolute counts:")
print(df_data['Label'].value_counts().to_string())
print(f"\nRelative proportions:")
for label, prop in df_data['Label'].value_counts(normalize=True).items():
    print(f"  {label:<30} {prop*100:>6.2f}%")
print(f"\nTotal unique labels: {df_data['Label'].nunique()}")
print(f"Total samples: {len(df_data):,}")

# Extract features and target
print("\n" + "-"*80)
print("FEATURE EXTRACTION")
print("-"*80)
X = df_data.drop('Label', axis=1).copy()
y = df_data['Label'].copy()

# Binarize labels: Benign=0, Attack=1
print("\nBinarizing labels (Benign=0, Attack=1)...")
y = y.map({'Benign': 0}).fillna(1).astype(int)
print(f"\nFeatures (X) shape: {X.shape}")
print(f"Target (y) shape: {y.shape}")
print(f"Features data type: {X.dtypes.value_counts().to_dict()}")
print(f"\nBinary class distribution:")
for class_label, count in pd.Series(y).value_counts().sort_index().items():
    class_name = "Benign" if class_label == 0 else "Attack"
    print(f"  Class {class_label} ({class_name}): {count:>10,} samples ({count/len(y)*100:>5.2f}%)")
print(f"\nClass imbalance ratio: {pd.Series(y).value_counts().max() / pd.Series(y).value_counts().min():.2f}:1")

# =============================================================================
# DATA SPLITTING
# =============================================================================
print("\n" + "="*80)
print("DATA SPLITTING")
print("="*80)

# Split: 75% train, 10% val, 15% test
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, 
    stratify=y,
    test_size=0.25,
    random_state=seed,
    shuffle=True
)

# Split temp into val and test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    stratify=y_temp,
    test_size=0.6,  # 0.6 of 25% = 15% test
    random_state=seed,
    shuffle=True
)

print(f"\nDataset split results:")
print(f"  Train set: {X_train.shape[0]:>10,} samples ({X_train.shape[0]/len(X)*100:>5.2f}%)")
print(f"  Val set:   {X_val.shape[0]:>10,} samples ({X_val.shape[0]/len(X)*100:>5.2f}%)")
print(f"  Test set:  {X_test.shape[0]:>10,} samples ({X_test.shape[0]/len(X)*100:>5.2f}%)")
print(f"\nClass distribution in splits:")
print(f"  Train - Benign: {(y_train==0).sum():>10,}, Attack: {(y_train==1).sum():>10,}")
print(f"  Val   - Benign: {(y_val==0).sum():>10,}, Attack: {(y_val==1).sum():>10,}")
print(f"  Test  - Benign: {(y_test==0).sum():>10,}, Attack: {(y_test==1).sum():>10,}")

# =============================================================================
# DATA SCALING
# =============================================================================
print("\n" + "="*80)
print("DATA SCALING (Z-SCORE NORMALIZATION)")
print("="*80)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print(f"Train set scaled: {X_train_scaled.shape}")
print(f"Val set scaled:   {X_val_scaled.shape}")
print(f"Test set scaled:  {X_test_scaled.shape}")

# Save scaler for inference
scaler_path = os.path.join(OUT_DIR, 'scaler.pkl')
joblib.dump(scaler, scaler_path)
print(f"💾 Scaler saved: {scaler_path}")

# Convert labels to numpy arrays
y_train_np = y_train.values
y_val_np = y_val.values
y_test_np = y_test.values

# =============================================================================
# VISUALIZATION UTILITIES
# =============================================================================

def plot_confusion_matrix(cm, model_name, save_path):
    """Plot confusion matrix heatmap."""
    plt.figure(figsize=(8, 6))
    
    # Normalize confusion matrix
    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    
    # Create annotations with both counts and percentages
    annot = np.empty_like(cm, dtype=object)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            annot[i, j] = f"{cm[i, j]:,}\n({cm_norm[i, j]*100:.1f}%)"
    
    sns.heatmap(cm_norm, annot=annot, fmt='', cmap='Blues', 
                xticklabels=['Benign', 'Attack'],
                yticklabels=['Benign', 'Attack'],
                cbar_kws={'label': 'Proportion'},
                linewidths=1, linecolor='gray')
    
    plt.title(f'{model_name} - Confusion Matrix', fontsize=14, fontweight='bold', pad=20)
    plt.ylabel('Actual Label', fontsize=12)
    plt.xlabel('Predicted Label', fontsize=12)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"    📊 Confusion matrix plot saved: {save_path}")
    plt.close()


def plot_roc_curve(labels, probs, model_name, save_path):
    """Plot ROC curve and save with AUC annotation."""
    fpr, tpr, _ = roc_curve(labels, probs)
    roc_auc = auc(fpr, tpr)

    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc*100:.2f}%)')
    plt.plot([0, 1], [0, 1], color='navy', lw=1, linestyle='--')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate', fontsize=12)
    plt.ylabel('True Positive Rate', fontsize=12)
    plt.title(f'{model_name} - ROC Curve', fontsize=14, fontweight='bold')
    plt.legend(loc='lower right')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"    🔍 ROC curve plot saved: {save_path} (AUC={roc_auc*100:.2f}%)")
    plt.close()


def plot_pr_curve(labels, probs, model_name, save_path):
    """Plot Precision-Recall curve and save with Average Precision (AP) annotation."""
    precision_vals, recall_vals, _ = precision_recall_curve(labels, probs)
    ap = average_precision_score(labels, probs)

    plt.figure(figsize=(8, 6))
    plt.plot(recall_vals, precision_vals, color='purple', lw=2, label=f'PR curve (AP = {ap*100:.2f}%)')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('Recall', fontsize=12)
    plt.ylabel('Precision', fontsize=12)
    plt.title(f'{model_name} - Precision-Recall Curve', fontsize=14, fontweight='bold')
    plt.legend(loc='lower left')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"    🔍 PR curve plot saved: {save_path} (AP={ap*100:.2f}%)")
    plt.close()


def plot_metrics_comparison(results, save_path):
    """Plot comparison of all models across different metrics."""
    metrics = ['accuracy', 'recall', 'f1', 'auc']
    metric_names = ['Accuracy', 'Detection Rate', 'F1 Score', 'AUC-ROC']
    model_names = list(results.keys())
    
    fig, ax = plt.subplots(figsize=(14, 7))
    
    x = np.arange(len(metrics))
    width = 0.8 / len(model_names)
    
    # Extended color palette for 6 models
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']
    
    for i, model_name in enumerate(model_names):
        color = colors[i % len(colors)]
        values = [results[model_name][metric] * 100 for metric in metrics]
        offset = (i - len(model_names)/2 + 0.5) * width
        bars = ax.bar(x + offset, values, width, label=model_name, color=color, alpha=0.8)
        
        # Add value labels on bars
        for bar in bars:
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height,
                   f'{height:.1f}%',
                   ha='center', va='bottom', fontsize=9)
    
    ax.set_xlabel('Metrics', fontsize=12, fontweight='bold')
    ax.set_ylabel('Score (%)', fontsize=12, fontweight='bold')
    ax.set_title('Model Performance Comparison', fontsize=14, fontweight='bold', pad=20)
    ax.set_xticks(x)
    ax.set_xticklabels(metric_names)
    ax.legend(fontsize=10, loc='lower right')
    ax.grid(True, alpha=0.3, axis='y')
    ax.set_ylim([0, 105])
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"\n📊 Model comparison plot saved: {save_path}")
    plt.close()


def plot_model_summary(results, save_path):
    """Create a comprehensive summary visualization."""
    fig = plt.figure(figsize=(16, 12))
    gs = fig.add_gridspec(3, 2, hspace=0.3, wspace=0.3)
    
    model_names = list(results.keys())
    # Extended color palette for 6 models
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']
    model_colors = [colors[i % len(colors)] for i in range(len(model_names))]
    
    # 1. F1 Score comparison (main metric)
    ax1 = fig.add_subplot(gs[0, :])
    f1_scores = [results[m]['f1'] * 100 for m in model_names]
    bars = ax1.barh(model_names, f1_scores, color=model_colors, alpha=0.8)
    for i, (bar, score) in enumerate(zip(bars, f1_scores)):
        ax1.text(score + 0.5, i, f'{score:.2f}%', va='center', fontsize=11, fontweight='bold')
    ax1.set_xlabel('F1 Score (%)', fontsize=12, fontweight='bold')
    ax1.set_title('F1 Score Comparison (Primary Metric)', fontsize=14, fontweight='bold')
    ax1.grid(True, alpha=0.3, axis='x')
    ax1.set_xlim([0, 105])
    
    # 2-5. Individual metric plots
    metrics = [('accuracy', 'Accuracy'), ('recall', 'Detection Rate'), 
               ('f1', 'F1 Score'), ('auc', 'AUC-ROC')]
    positions = [(1, 0), (1, 1), (2, 0), (2, 1)]
    
    for (metric, title), pos in zip(metrics, positions):
        ax = fig.add_subplot(gs[pos])
        values = [results[m][metric] * 100 for m in model_names]
        bars = ax.bar(range(len(model_names)), values, color=model_colors, alpha=0.8)
        
        for bar, val in zip(bars, values):
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height,
                   f'{val:.1f}%', ha='center', va='bottom', fontsize=9)
        
        ax.set_xticks(range(len(model_names)))
        ax.set_xticklabels(model_names, rotation=45, ha='right')
        ax.set_ylabel('Score (%)', fontsize=10)
        ax.set_title(title, fontsize=12, fontweight='bold')
        ax.grid(True, alpha=0.3, axis='y')
        ax.set_ylim([0, 105])
    
    plt.suptitle('Model Performance Summary', fontsize=16, fontweight='bold', y=0.995)
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"📊 Model summary plot saved: {save_path}")
    plt.close()


# =============================================================================
# TRAINING AND EVALUATION UTILITIES
# =============================================================================

def to_numpy(arr):
    """Convert cupy/cudf arrays to numpy if needed."""
    if USE_GPU:
        try:
            if hasattr(arr, 'get'):
                return arr.get()
            elif hasattr(arr, 'to_numpy'):
                return arr.to_numpy()
        except:
            pass
    return np.asarray(arr)

def train_sklearn_model(model, model_name, X_train, y_train, X_test, y_test, save_path=None):
    """
    Train and evaluate a scikit-learn or cuML model with GPU acceleration.
    
    Parameters:
        model: sklearn/cuML model instance
        model_name: Name of the model for display
        X_train: Training features
        y_train: Training labels
        X_test: Test features
        y_test: Test labels
        save_path: Path to save the model (optional)
    
    Returns:
        Dictionary with evaluation results
    """
    print(f"\n{'='*80}")
    print(f"TRAINING: {model_name} {'[GPU]' if USE_GPU else '[CPU]'}")
    print(f"{'='*80}")
    print(f"Training samples: {len(y_train):,}")
    print(f"Test samples: {len(y_test):,}")
    print("-" * 80)
    
    # Train
    start_time = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - start_time
    print(f"    Training time: {train_time:.2f} seconds")
    
    # Predict
    start_pred = time.time()
    predictions = to_numpy(model.predict(X_test))
    pred_time = time.time() - start_pred
    print(f"    Prediction time: {pred_time:.4f} seconds")
    
    # Convert y_test to numpy for metrics
    y_test_np = to_numpy(y_test)
    
    # Get probabilities for AUC
    if hasattr(model, 'predict_proba'):
        probs = to_numpy(model.predict_proba(X_test))
        if probs.shape[1] > 1:
            probabilities = probs[:, 1]
        else:
            probabilities = probs[:, 0]
    else:
        probabilities = predictions.astype(float)
    
    # Calculate metrics
    accuracy = accuracy_score(y_test_np, predictions)
    precision = precision_score(y_test_np, predictions, zero_division=0)
    recall = recall_score(y_test_np, predictions, zero_division=0)
    f1 = f1_score(y_test_np, predictions, zero_division=0)
    auc_score = roc_auc_score(y_test_np, probabilities)
    
    # Save model
    if save_path:
        joblib.dump(model, save_path)
        print(f"\n💾  Model saved: {save_path}")
    
    print(f"\n📊  {model_name} Test Set Evaluation:")
    print("-" * 80)
    print(f"    Accuracy:       {accuracy*100:>6.2f}%")
    print(f"    Detection Rate: {recall*100:>6.2f}%")
    print(f"    F1 Score:       {f1*100:>6.2f}%")
    print(f"    AUC-ROC:        {auc_score*100:>6.2f}%")
    print("\n    Confusion Matrix:")
    cm = confusion_matrix(y_test_np, predictions)
    print(f"                Predicted")
    print(f"                Benign  Attack")
    print(f"    Actual Benign  {cm[0,0]:>6,}  {cm[0,1]:>6,}")
    print(f"           Attack  {cm[1,0]:>6,}  {cm[1,1]:>6,}")
    print("\n    Classification Report:")
    print(classification_report(y_test_np, predictions, 
                                target_names=['Benign', 'Attack'], digits=4))
    
    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'detection_rate': recall,  # Detection rate = Recall for attack class
        'f1': f1,
        'auc': auc_score,
        'predictions': predictions,
        'probabilities': probabilities,
        'labels': y_test_np,
        'train_time': train_time,
        'model': model
    }


def train_apollon_model(X_train, y_train, X_test, y_test, n_clusters=2, save_path=None):
    """
    Train and evaluate the Apollon MAB model.
    
    Parameters:
        X_train: Training features
        y_train: Training labels
        X_test: Test features
        y_test: Test labels
        n_clusters: Number of clusters for MAB
        save_path: Path to save the model (optional)
    
    Returns:
        Dictionary with evaluation results
    """
    print(f"\n{'='*80}")
    print(f"TRAINING: Apollon (Multi-Armed Bandit with Thompson Sampling)")
    print(f"{'='*80}")
    print(f"Training samples: {len(y_train):,}")
    print(f"Test samples: {len(y_test):,}")
    print(f"Number of clusters: {n_clusters}")
    print(f"Number of arms (classifiers): 5 (RF, DT, NB, LR, MLP)")
    print("-" * 80)
    
    # Initialize and train Apollon
    apollon = MultiArmedBanditThompsonSampling(n_arms=5, n_clusters=n_clusters, seed=seed)
    apollon.train(X_train, y_train)
    
    # Predict
    start_pred = time.time()
    predictions, selected_arms = apollon.predict(X_test)
    pred_time = time.time() - start_pred
    print(f"\n    Prediction time: {pred_time:.4f} seconds")
    
    # Get probabilities
    probabilities = apollon.predict_proba(X_test)
    
    # Convert to numpy for sklearn metrics (handles cupy/cudf arrays)
    predictions = to_numpy(predictions)
    probabilities = to_numpy(probabilities)
    selected_arms = to_numpy(selected_arms)
    y_test_np = to_numpy(y_test)
    
    # Calculate metrics
    accuracy = accuracy_score(y_test_np, predictions)
    precision = precision_score(y_test_np, predictions, zero_division=0)
    recall = recall_score(y_test_np, predictions, zero_division=0)
    f1 = f1_score(y_test_np, predictions, zero_division=0)
    auc_score = roc_auc_score(y_test_np, probabilities)
    
    # Arm selection statistics
    print(f"\n    Arm Selection Statistics:")
    for arm_idx, arm_name in enumerate(apollon.arm_names):
        count = np.sum(selected_arms == arm_idx)
        print(f"      {arm_name}: {count:,} ({count/len(y_test_np)*100:.2f}%)")
    
    # Save model
    if save_path:
        joblib.dump(apollon, save_path)
        print(f"\n💾  Model saved: {save_path}")
    
    print(f"\n📊  Apollon Test Set Evaluation:")
    print("-" * 80)
    print(f"    Accuracy:       {accuracy*100:>6.2f}%")
    print(f"    Detection Rate: {recall*100:>6.2f}%")
    print(f"    F1 Score:       {f1*100:>6.2f}%")
    print(f"    AUC-ROC:        {auc_score*100:>6.2f}%")
    print("\n    Confusion Matrix:")
    cm = confusion_matrix(y_test_np, predictions)
    print(f"                Predicted")
    print(f"                Benign  Attack")
    print(f"    Actual Benign  {cm[0,0]:>6,}  {cm[0,1]:>6,}")
    print(f"           Attack  {cm[1,0]:>6,}  {cm[1,1]:>6,}")
    print("\n    Classification Report:")
    print(classification_report(y_test_np, predictions, 
                                target_names=['Benign', 'Attack'], digits=4))
    
    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'detection_rate': recall,
        'f1': f1,
        'auc': auc_score,
        'predictions': predictions,
        'probabilities': probabilities,
        'labels': y_test_np,
        'train_time': apollon.training_time,
        'model': apollon,
        'selected_arms': selected_arms
    }


# =============================================================================
# MODEL TRAINING LOOP
# =============================================================================
print("\n" + "="*80)
print("MODEL POOL TRAINING")
print("="*80)

results = {}
trained_models = {}

# MLP Model (sklearn)
if MODELS_TO_TRAIN.get('mlp_model', False):
    mlp = MLPClassifier(
        hidden_layer_sizes=(32,),
        max_iter=200,
        verbose=False,
        random_state=seed,
        batch_size=200,
        early_stopping=True,
        activation='tanh',
        solver='adam'
    )
    eval_results = train_sklearn_model(
        mlp, 'MLP (Multi-Layer Perceptron)',
        X_train_scaled, y_train_np, X_test_scaled, y_test_np,
        save_path=os.path.join(OUT_DIR, 'mlp_model.pkl')
    )
    results['MLP'] = eval_results
    trained_models['MLP'] = eval_results['model']
    
    # Generate plots
    cm = confusion_matrix(eval_results['labels'], eval_results['predictions'])
    plot_confusion_matrix(cm, 'MLP', os.path.join(PLOT_DIR, 'mlp_confusion_matrix.png'))
    plot_roc_curve(eval_results['labels'], eval_results['probabilities'], 'MLP', os.path.join(PLOT_DIR, 'mlp_roc_curve.png'))
    plot_pr_curve(eval_results['labels'], eval_results['probabilities'], 'MLP', os.path.join(PLOT_DIR, 'mlp_pr_curve.png'))

# Naive Bayes Model
if MODELS_TO_TRAIN.get('nb_model', False):
    nb = GaussianNB()
    eval_results = train_sklearn_model(
        nb, 'NB (Naive Bayes)',
        X_train_scaled, y_train_np, X_test_scaled, y_test_np,
        save_path=os.path.join(OUT_DIR, 'nb_model.pkl')
    )
    results['NB'] = eval_results
    trained_models['NB'] = eval_results['model']
    
    # Generate plots
    cm = confusion_matrix(eval_results['labels'], eval_results['predictions'])
    plot_confusion_matrix(cm, 'NB', os.path.join(PLOT_DIR, 'nb_confusion_matrix.png'))
    plot_roc_curve(eval_results['labels'], eval_results['probabilities'], 'NB', os.path.join(PLOT_DIR, 'nb_roc_curve.png'))
    plot_pr_curve(eval_results['labels'], eval_results['probabilities'], 'NB', os.path.join(PLOT_DIR, 'nb_pr_curve.png'))

# Random Forest Model
if MODELS_TO_TRAIN.get('rf_model', False):
    if USE_GPU:
        rf = RandomForestClassifier(n_estimators=100, random_state=seed)
    else:
        rf = RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=seed)
    eval_results = train_sklearn_model(
        rf, 'RF (Random Forest)',
        X_train_scaled, y_train_np, X_test_scaled, y_test_np,
        save_path=os.path.join(OUT_DIR, 'rf_model.pkl')
    )
    results['RF'] = eval_results
    trained_models['RF'] = eval_results['model']
    
    # Generate plots
    cm = confusion_matrix(eval_results['labels'], eval_results['predictions'])
    plot_confusion_matrix(cm, 'RF', os.path.join(PLOT_DIR, 'rf_confusion_matrix.png'))
    plot_roc_curve(eval_results['labels'], eval_results['probabilities'], 'RF', os.path.join(PLOT_DIR, 'rf_roc_curve.png'))
    plot_pr_curve(eval_results['labels'], eval_results['probabilities'], 'RF', os.path.join(PLOT_DIR, 'rf_pr_curve.png'))

# Decision Tree Model
if MODELS_TO_TRAIN.get('dt_model', False):
    if USE_GPU:
        dt = DecisionTreeClassifier()
    else:
        dt = DecisionTreeClassifier(random_state=seed)
    eval_results = train_sklearn_model(
        dt, 'DT (Decision Tree)',
        X_train_scaled, y_train_np, X_test_scaled, y_test_np,
        save_path=os.path.join(OUT_DIR, 'dt_model.pkl')
    )
    results['DT'] = eval_results
    trained_models['DT'] = eval_results['model']
    
    # Generate plots
    cm = confusion_matrix(eval_results['labels'], eval_results['predictions'])
    plot_confusion_matrix(cm, 'DT', os.path.join(PLOT_DIR, 'dt_confusion_matrix.png'))
    plot_roc_curve(eval_results['labels'], eval_results['probabilities'], 'DT', os.path.join(PLOT_DIR, 'dt_roc_curve.png'))
    plot_pr_curve(eval_results['labels'], eval_results['probabilities'], 'DT', os.path.join(PLOT_DIR, 'dt_pr_curve.png'))

# Logistic Regression Model
if MODELS_TO_TRAIN.get('lr_model', False):
    if USE_GPU:
        lr = LogisticRegression(max_iter=1000)
    else:
        lr = LogisticRegression(max_iter=1000, random_state=seed)
    eval_results = train_sklearn_model(
        lr, 'LR (Logistic Regression)',
        X_train_scaled, y_train_np, X_test_scaled, y_test_np,
        save_path=os.path.join(OUT_DIR, 'lr_model.pkl')
    )
    results['LR'] = eval_results
    trained_models['LR'] = eval_results['model']
    
    # Generate plots
    cm = confusion_matrix(eval_results['labels'], eval_results['predictions'])
    plot_confusion_matrix(cm, 'LR', os.path.join(PLOT_DIR, 'lr_confusion_matrix.png'))
    plot_roc_curve(eval_results['labels'], eval_results['probabilities'], 'LR', os.path.join(PLOT_DIR, 'lr_roc_curve.png'))
    plot_pr_curve(eval_results['labels'], eval_results['probabilities'], 'LR', os.path.join(PLOT_DIR, 'lr_pr_curve.png'))

# Apollon Model (Multi-Armed Bandit with Thompson Sampling)
if MODELS_TO_TRAIN.get('apollon_model', False):
    eval_results = train_apollon_model(
        X_train_scaled, y_train_np, X_test_scaled, y_test_np,
        n_clusters=2,
        save_path=os.path.join(OUT_DIR, 'apollon_model.pkl')
    )
    results['Apollon'] = eval_results
    trained_models['Apollon'] = eval_results['model']
    
    # Generate plots
    cm = confusion_matrix(eval_results['labels'], eval_results['predictions'])
    plot_confusion_matrix(cm, 'Apollon', os.path.join(PLOT_DIR, 'apollon_confusion_matrix.png'))
    plot_roc_curve(eval_results['labels'], eval_results['probabilities'], 'Apollon', os.path.join(PLOT_DIR, 'apollon_roc_curve.png'))
    plot_pr_curve(eval_results['labels'], eval_results['probabilities'], 'Apollon', os.path.join(PLOT_DIR, 'apollon_pr_curve.png'))

# =============================================================================
# VISUALIZATION SUMMARY
# =============================================================================
if results:
    print("\n" + "="*80)
    print("GENERATING COMPARISON VISUALIZATIONS")
    print("="*80)
    
    # Create comparison plots
    plot_metrics_comparison(results, os.path.join(PLOT_DIR, 'model_comparison.png'))
    plot_model_summary(results, os.path.join(PLOT_DIR, 'model_summary.png'))
    
    print(f"\n✅ All plots saved to: {PLOT_DIR}")
    print(f"   - Individual confusion matrices ({len(results)} models)")
    print(f"   - Model comparison plot")
    print(f"   - Comprehensive model summary plot")

# =============================================================================
# FINAL SUMMARY (TABLE 2 FORMAT)
# =============================================================================
print("\n" + "="*80)
print("TRAINING SUMMARY")
print("="*80)

if results:
    print("\n" + "="*80)
    print("Table 2: Results of the traditional network traffic and attacks scenario")
    print("         on the CIC-IDS-2017 dataset.")
    print("="*80)
    
    # Print header matching Table 2 format
    print(f"\n{'Metrics':<15}", end="")
    print("CIC-IDS-2017")
    print("-" * 80)
    
    # Column headers
    model_order = ['MLP', 'NB', 'RF', 'DT', 'LR', 'Apollon']
    available_models = [m for m in model_order if m in results]
    
    print(f"{'Metrics':<15}", end="")
    for model in available_models:
        print(f"{model:>12}", end="")
    print()
    print("-" * 80)
    
    # Accuracy row
    print(f"{'Accuracy':<15}", end="")
    for model in available_models:
        print(f"{results[model]['accuracy']:>12.4f}", end="")
    print()
    
    # Detection rate row (Recall)
    print(f"{'Detection rate':<15}", end="")
    for model in available_models:
        print(f"{results[model]['recall']:>12.4f}", end="")
    print()
    
    # F1 row
    print(f"{'F1':<15}", end="")
    for model in available_models:
        print(f"{results[model]['f1']:>12.4f}", end="")
    print()
    
    # AUC row
    print(f"{'AUC':<15}", end="")
    for model in available_models:
        print(f"{results[model]['auc']:>12.4f}", end="")
    print()
    
    print("-" * 80)
    
    # Extended comparison table
    print("\n\nDetailed Model Comparison:")
    print("-" * 100)
    print(f"{'Model':<15} {'Accuracy':<12} {'Detection Rate':<15} {'F1 Score':<12} {'AUC-ROC':<12} {'Train Time':<12}")
    print("-" * 100)
    
    for model_name in available_models:
        res = results[model_name]
        train_time = f"{res.get('train_time', 0):.2f}s"
        print(f"{model_name:<15} "
              f"{res['accuracy']*100:>10.2f}% "
              f"{res['recall']*100:>13.2f}% "
              f"{res['f1']*100:>10.2f}% "
              f"{res['auc']*100:>10.2f}% "
              f"{train_time:>10}")
    
    print("-" * 100)
    
    # Find best model by F1
    best_model_name = max(results, key=lambda k: results[k]['f1'])
    best_f1 = results[best_model_name]['f1']
    print(f"\n🏆 Best Model by F1 Score: {best_model_name} (F1: {best_f1*100:.2f}%)")
    
    # Find best model by Accuracy
    best_acc_model = max(results, key=lambda k: results[k]['accuracy'])
    best_acc = results[best_acc_model]['accuracy']
    print(f"🏆 Best Model by Accuracy: {best_acc_model} (Accuracy: {best_acc*100:.2f}%)")
    
    print(f"\n✅ All models saved to: {OUT_DIR}")
else:
    print("\n⚠️ No models were trained. Check MODELS_TO_TRAIN configuration.")

print("\n" + "="*80)
print("TRAINING COMPLETE")
print("="*80)
if USE_GPU:
    print(f"🚀 Hardware: GPU (RAPIDS cuML)")
else:
    print(f"🔧 Hardware: CPU (sklearn)")
print("="*80 + "\n")






Looking in indexes: https://pypi.org/simple, https://pypi.nvidia.com
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 25.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 153.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 218.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 445.9/445.9 MB 86.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 192.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 151.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 228.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.6/58.6 MB 186.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 193.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 849.8/849.8 kB 236.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.4/138.4 kB 135.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5

/usr/local/lib/python3.11/dist-packages/cupy/_environment.py:596: UserWarning: 
--------------------------------------------------------------------------------

  CuPy may not function correctly because multiple CuPy packages are installed
  in your environment:

    cupy-cuda11x, cupy-cuda12x

  Follow these steps to resolve this issue:

    1. For all packages listed above, run the following command to remove all
       existing CuPy installations:

         $ pip uninstall <package_name>

      If you previously installed CuPy via conda, also run the following:

         $ conda uninstall cupy

    2. Install the appropriate CuPy package.
       Refer to the Installation Guide for detailed instructions.

         https://docs.cupy.dev/en/stable/install.html

--------------------------------------------------------------------------------

  warnings.warn(f'''



⚠️ RAPIDS cuML not available - using CPU sklearn
   Reason: RuntimeError: RAPIDS requires Compute Capability >= 7.0, detected 6.0. Falling back to CPU sklearn.
   If you want RAPIDS GPU acceleration: use a Volta+ GPU (T4/V100/A100, CC>=7.0).
   Otherwise: keep sklearn/pandas (CPU) and everything will still run.

🔧 Using CPU models (sklearn)


DATA LOADING AND PREPROCESSING
Loading 7 CSV files from /kaggle/input/cic-ids2017/MachineLearningCVE...
  [1/7] Loading: Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
      Initial shape: (225745, 79)
      Memory usage: 147.66 MB
  [2/7] Loading: Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
      Initial shape: (286467, 79)
      Memory usage: 187.99 MB
  [3/7] Loading: Friday-WorkingHours-Morning.pcap_ISCX.csv
      Initial shape: (191033, 79)
      Memory usage: 125.15 MB
  [4/7] Loading: Monday-WorkingHours.pcap_ISCX.csv
      Initial shape: (529918, 79)
      Memory usage: 347.19 MB
  [5/7] Loading: Thursday-WorkingHours-Afternoon